In [1]:
# set seed 
import torch
import random
import numpy as np 
import transformers
import wandb


SEED = 42 # for reproduciple 
BLOCK_SIZE = 2048 # total token for each chunk/ each example in packing

def set_seed(seed = 42):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed(seed)
    transformers.set_seed(seed)

set_seed(SEED)
wandb.init(project="LLM-Learning", name="qwen-medical-cpt", resume="allow")

Skipping import of cpp extensions due to incompatible torch version. Please upgrade to torch >= 2.11.0 (found 2.10.0+cu128).
wandb: [wandb.login()] Loaded credentials for https://api.wandb.ai from /home/pc5070ti/.netrc.
wandb: Currently logged in as: nguyentranhoangnhan18 (nguyentranhoangnhan18-ton-duc-thang-university) to https://api.wandb.ai. Use `wandb login --relogin` to force relogin


# Data Preprocessing

## Load dataset

In [2]:
from datasets import load_dataset


# Dataset 1 — wikidoc (~10k samples)
wikidoc = load_dataset("medalpaca/medical_meadow_wikidoc", split="train")

# Dataset 2 — PubMedQA (~200k samples)
pubmedqa = load_dataset("qiaojin/PubMedQA", "pqa_artificial", split="train")

In [3]:
print(wikidoc)
print(pubmedqa)

Dataset({
    features: ['input', 'output', 'instruction'],
    num_rows: 10000
})
Dataset({
    features: ['pubid', 'question', 'context', 'long_answer', 'final_decision'],
    num_rows: 211269
})


# Combine datasets

In [5]:
# ghép cho wikidoc (lay output)
def extract_wikidoc(example):
    text = f"{example['input']}: {example['output']}"
    return {"text": text}

# ghép cho pubmedqa (context + long_answer)
def extract_pubmed(example):
    # vi contexts la list cac doan -> join lai thanh 1 doan cach nhau bang dau cach
    contexts = " ".join(example["context"]["contexts"])
    long_answer = example["long_answer"]

    text = f"{contexts}\n{long_answer}"
    return {"text": text}



wikidoc_text = wikidoc.map(extract_wikidoc, remove_columns=wikidoc.column_names)
pubmedqa_text = pubmedqa.map(extract_pubmed, remove_columns=pubmedqa.column_names)

print(wikidoc_text)
print(wikidoc_text[0])
print("---")
print(pubmedqa_text)
print(pubmedqa_text[0])


Dataset({
    features: ['text'],
    num_rows: 10000
})
{'text': "Can you provide an overview of the lung's squamous cell carcinoma?: Squamous cell carcinoma of the lung may be classified according to the WHO histological classification system into 4 main types: papillary, clear cell, small cell, and basaloid."}
---
Dataset({
    features: ['text'],
    num_rows: 211269
})
{'text': 'Chronic rhinosinusitis (CRS) is a heterogeneous disease with an uncertain pathogenesis. Group 2 innate lymphoid cells (ILC2s) represent a recently discovered cell population which has been implicated in driving Th2 inflammation in CRS; however, their relationship with clinical disease characteristics has yet to be investigated. The aim of this study was to identify ILC2s in sinus mucosa in patients with CRS and controls and compare ILC2s across characteristics of disease. A cross-sectional study of patients with CRS undergoing endoscopic sinus surgery was conducted. Sinus mucosal biopsies were obtained dur

In [6]:
# concat 2 cái data lại với nhau
from datasets import concatenate_datasets

combined_datasets = concatenate_datasets([wikidoc_text, pubmedqa_text])

In [7]:
print(combined_datasets)

Dataset({
    features: ['text'],
    num_rows: 221269
})


In [8]:

combined_datasets = combined_datasets.shuffle(seed=SEED)

# take small subset because it too large, take too much computation cost
# combined_datasets = combined_datasets.select(range(5000))

print(combined_datasets[0])
print(combined_datasets[1])
print(combined_datasets)

{'text': 'How can secondary prevention be employed in the management of osteoporosis?: Effective measures for the secondary prevention of osteoporosis include pharmacological therapy and also lifestyle modification.\nThe primary goal for the treatment of osteoporosis is to reduce longtime fracture risk in patients. Increasing bone mineral density (BMD) in response to the treatment is far less important than improvement of clinical aspects of osteoporosis, i.e., osteoporotic fracture. Therefore, most of the drugs efficacy is measured by the extent they improve the fracture risk instead of increasing BMD.  During the treatment, if a single fracture happens, it does not necessarily indicate treatment failure or the need to be started on an alternative treatment or patient referral to a specialist.  Calcium and vitamin D supplementation have been found to be effective in reducing the long term fracture risk, significantly. In order to suggest the people to use vitamin D and calcium supplem

## Tokenize

In [9]:
# tokenize
from transformers import AutoTokenizer

tokenizer = AutoTokenizer.from_pretrained("Qwen/Qwen2.5-0.5B")

def tokenize(examples): # examples lúc này là nguyên cái batch vì batch = true
    return tokenizer(
        examples["text"], # lấy cái list chứa batch của text
        # tokenizer của hugging face tự nhận 1 list -> trả về dict của list
        truncation=False, # cắt bớt nếu vượt quá số lượng max_token
        padding=False, # đệm padding
        # -> ko vì lát dùng packing
    )

tokenized = combined_datasets.map(
    tokenize, #
    batched=True, # tokenize nhiều sample cùng lúc
    # nếu batch = false thì example là 1 sample= {"text": "..."}
    # batch = true thì là dict of a list examples = {"text": ["...", "..."]}
    batch_size=1000,
    remove_columns=["text"],
    # loại cột text ra vì ko cần, vì map thì nó là tạo ra 2 cái column mới (với cái này là input_ids, attention_mask)
)

print(tokenized)
print(tokenized[0])

Dataset({
    features: ['input_ids', 'attention_mask'],
    num_rows: 221269
})
{'input_ids': [4340, 646, 14246, 26248, 387, 19446, 304, 279, 6240, 315, 51268, 88666, 10704, 4820, 46923, 10953, 369, 279, 14246, 26248, 315, 51268, 88666, 10704, 2924, 35349, 5729, 15069, 323, 1083, 18899, 17030, 624, 785, 6028, 5795, 369, 279, 6380, 315, 51268, 88666, 10704, 374, 311, 7949, 35404, 58804, 5214, 304, 6835, 13, 73440, 17240, 24136, 17457, 320, 33, 6076, 8, 304, 2033, 311, 279, 6380, 374, 3041, 2686, 2989, 1091, 15673, 315, 14490, 13566, 315, 51268, 88666, 10704, 11, 600, 1734, 2572, 51268, 88666, 14212, 58804, 13, 15277, 11, 1429, 315, 279, 10975, 40165, 374, 16878, 553, 279, 12818, 807, 7269, 279, 58804, 5214, 4518, 315, 7703, 425, 6076, 13, 220, 11954, 279, 6380, 11, 421, 264, 3175, 58804, 8573, 11, 432, 1558, 537, 14312, 13216, 6380, 7901, 476, 279, 1184, 311, 387, 3855, 389, 458, 10555, 6380, 476, 8720, 44780, 311, 264, 23753, 13, 220, 95004, 323, 27071, 422, 72696, 614, 1012, 1730, 31

## Packing

In [13]:
# PACKING
# ghép các sample lại với nhau để sao cho nó = đúng max token

from datasets import Dataset

def pack_tokens(examples, block_size=2048):
    # examples["input_ids"] là list of list


    # 1. concat tất cả token thành 1 list dài
    all_tokens = []
    for ids in examples["input_ids"]:
        all_tokens.extend(ids)
        # token EOS end of sentence de ngat giua cac sample
        all_tokens.append(tokenizer.eos_token_id)

    # 2. chia thành các chunks có đúng block_size
    # chunks_input_ids = []
    # chunks_attentions_mask = []

    # # it like range(0, len() - 1, 1) but here 1 is blocksize
    # for i in range(0, len(all_tokens) - block_size, block_size):
    #     chunk = all_tokens[i: i + block_size]
    #     chunks_input_ids.append(chunk) # list of list, each list is len of block-size
    #     chunks_attentions_mask.append([1] * block_size) # because al token are real token, not padding, and this mean assign 1 * blocksize

    """
    sai logic, ko con hop le, range se xet thieu last item 
    """
    

    # 2. new logic 
    usable_chunk = len(all_tokens) // block_size 
    usable_length = usable_chunk * block_size 
    dropped_token = len(all_tokens) - usable_length
   
    print(f"Total tokens: {len(all_tokens)}") 
    print(f"Usable tokens: {usable_length}")
    print(f"Dropped tokens: {dropped_token}")

    usable_tokens = all_tokens[:usable_length] 

    chunk_input_ids = [
        usable_tokens[i: i + block_size] for i in range(0, usable_length, block_size)
    ]
    
    chunk_attention_mask = [[1] * block_size for _ in chunk_input_ids]
    

    return Dataset.from_dict({
        "input_ids": chunk_input_ids,
        "attention_mask": chunk_attention_mask,
    })

# packed = tokenized.map(
#     pack_tokens,
#     batched=True,
#     batch_size=100000,
#     remove_columns=tokenized.column_names,
# )

"""
ko sử dụng map bằng batch, vì lý do là nếu chia batch, dữ liệu sẽ bị phân chia ra thành các batch, vì thế lúc này logic gom toàn bộ sẽ ko đúng nữa
thay vào đó nó sẽ gom theo cụm batch
"""

# new logic 
packed = pack_tokens(tokenized, block_size=BLOCK_SIZE)


print(packed)
print(packed[0]["input_ids"][:10])
print(len(packed[0]["input_ids"]))
print(len(packed[0]["attention_mask"]))

Total tokens: 83318218
Usable tokens: 83316736
Dropped tokens: 1482
Dataset({
    features: ['input_ids', 'attention_mask'],
    num_rows: 40682
})
[4340, 646, 14246, 26248, 387, 19446, 304, 279, 6240, 315]
2048
2048


# split train test 

In [14]:
split = packed.train_test_split(test_size=0.05, seed=SEED)
train_dataset = split["train"]
eval_dataset = split["test"]

## Data Collator

In [15]:
# tạo labels từ input_ids, nó thục hiện shift 1 token sang trái
# input_ids: [50, 98, 1234, 567, ...]
# labels:    [98, 1234, 567, ???, ...] shift 1 token sang trái

from transformers import DataCollatorForLanguageModeling

data_collator = DataCollatorForLanguageModeling(
    tokenizer=tokenizer, # mục đích để lấy tokenizer.pad_token_id
    # thục ra sau khi packing thì ko cần padding, nhưng đó là tham số bắt buộc nên vẫn phải truyền vào
    mlm=False, # causal LM, Không phải masked LM
)

# sự khác nhau giữa causal LM (GPT style) và masked LM (BERT style)
# masked che ngẫu nhiên, xong có thể nhin trái hoặc phải để predit , loss tính trên tokn bị mask
# causal là che hết phải, chỉ nhìn trái predict phải, loss ính trên toàn bộ token


## load Model  

In [19]:
import torch
from transformers import AutoModelForCausalLM

model = AutoModelForCausalLM.from_pretrained(
    "Qwen/Qwen2.5-0.5B",
    torch_dtype=torch.bfloat16, # bfloat giu nguyen phan exponent cuar float32 chi thay doi man mantissa (precision) xuong chinh xac 4 chu so thay vi 8 chu so cuar float32, trach viec gradient qua lon gay Nan
)

total_params = sum(p.numel() for p in model.parameters())
print(f"total params: {total_params}")


Loading weights:   0%|          | 0/290 [00:00<?, ?it/s]

total params: 494032768


## TrainingArguments

In [20]:
from transformers import TrainingArguments

args = TrainingArguments(
    output_dir="qwen-medical-pretrained",

    # batch_size
    per_device_train_batch_size=1, # 1 batch = 4 chunks
    per_device_eval_batch_size=1,
    gradient_accumulation_steps=32, #   1 step = 32 batches
    eval_accumulation_steps=8, # tích lũy trong gpu rồi đưa sanng cpu 
    # bình thường thì hết 1 batch là update, còn cái này nó sẽ tích lũy lại rồi mới up

    # eval 
    eval_strategy="steps",
    eval_steps=100,

    # learning rate
    learning_rate=2e-5,
    lr_scheduler_type="cosine",
    warmup_steps=60, # warm up trong 60 steps (tương đương 5% total training steps)


    # Duration
    num_train_epochs=1,

    # Save checkpoint
    save_strategy="steps",
    load_best_model_at_end=True,
    metric_for_best_model="eval_loss",
    greater_is_better=False,
    save_steps=100, # steps số lần model update weight
    save_total_limit=3,
    

    # logging
    logging_steps=5,

    # Precision
    bf16=True,

    # Memory
    gradient_checkpointing=True,
    report_to="wandb",
    # run_name="qwen-medical-cpt",  # tên run trên dashboard

)

## Trainer

In [21]:
from transformers import Trainer

trainer = Trainer(
    model=model,
    args=args,
    train_dataset=train_dataset,
    eval_dataset=eval_dataset,
    data_collator=data_collator,
)


trainer.train()

Step,Training Loss,Validation Loss
100,2.012542,2.019126
200,2.005186,2.004130
300,1.979049,1.995214
400,1.972278,1.989007
500,1.990071,1.984694
600,1.986733,1.981762
700,1.977820,1.980037
800,1.956943,1.979060
900,1.993497,1.978603
1000,1.958676,1.978450


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

...

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

There were missing keys in the checkpoint model loaded: ['lm_head.weight'].


TrainOutput(global_step=1208, training_loss=1.9902510287745898, metrics={'train_runtime': 7583.6448, 'train_samples_per_second': 5.096, 'train_steps_per_second': 0.159, 'total_flos': 1.6996378625389363e+17, 'train_loss': 1.9902510287745898, 'epoch': 1.0})

In [22]:
# save
trainer.save_model("qwen-medical-pretrained")
tokenizer.save_pretrained("qwen-medical-pretrained")

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

('qwen-medical-pretrained/tokenizer_config.json',
 'qwen-medical-pretrained/chat_template.jinja',
 'qwen-medical-pretrained/tokenizer.json')

# Test Model 

In [ ]:
# Compare Zero-shot vs CPT (Continue PreTraining)
from transformers import AutoModelForCausalLM, AutoTokenizer
import torch 

# 1. Define medical prompts to test
medical_prompts = [
    "What are the symptoms of diabetes?",
    "How to treat hypertension?",
    "Describe the pathophysiology of heart failure."
]

# 2. Load ZERO-SHOT base model 
print("ZERO-SHOT MODEL (Base - No Medical Training)")
zero_shot_model = AutoModelForCausalLM.from_pretrained(
    "Qwen/Qwen2.5-0.5B",  
    torch_dtype=torch.bfloat16,  # Sử dụng bfloat16 để tiết kiệm VRAM
)
zero_shot_tokenizer = AutoTokenizer.from_pretrained("Qwen/Qwen2.5-0.5B")

# 3. Load CPT model (trained - đã train trên medical data)
print("CPT MODEL (Trained on Medical Data)")
cpt_model = AutoModelForCausalLM.from_pretrained(
    "qwen-medical-pretrained",  # Tên thư mục model đã save
    torch_dtype=torch.bfloat16,
)
cpt_tokenizer = AutoTokenizer.from_pretrained("qwen-medical-pretrained")

# 4. Loop qua từng prompt y tế
for prompt in medical_prompts: 
    print("\n" + "-"*80)
    print(f"PROMPT: {prompt}")
    print("-"*80)
    
    print("\n[ZERO-SHOT RESPONSE]:")
    # Tokenize: convert text thành token IDs
    input_ids_zero = zero_shot_tokenizer(prompt, return_tensors="pt")
    # Generate: tạo response từ prompt
    output_ids_zero = zero_shot_model.generate(
        **input_ids_zero,  # Unpack dict để truyền input_ids, attention_mask
        max_length=150,  # Độ dài max của response
        temperature=0.7,  # Điều khiển độ random (0.7 = trung bình)
        top_p=0.9,  # Nucleus sampling: chỉ chọn từ top 90% probability
    )
    # Decode: convert token IDs thành text
    text_zero = zero_shot_tokenizer.decode(output_ids_zero[0], skip_special_tokens=True)
    print(text_zero)

    print("\n[CPT RESPONSE]:")
    # Tokenize
    input_ids_cpt = cpt_tokenizer(prompt, return_tensors="pt")
    # Generate
    output_ids_cpt = cpt_model.generate(
        **input_ids_cpt,
        max_length=150,
        temperature=0.7,
        top_p=0.9,
    )
    # Decode
    text_cpt = cpt_tokenizer.decode(output_ids_cpt[0], skip_special_tokens=True)
    print(text_cpt)

ZERO-SHOT MODEL (Base - No Medical Training)


Loading weights:   0%|          | 0/290 [00:00<?, ?it/s]

CPT MODEL (Trained on Medical Data)


Loading weights:   0%|          | 0/290 [00:00<?, ?it/s]

The following generation flags are not valid and may be ignored: ['temperature', 'top_p']. Set `TRANSFORMERS_VERBOSITY=info` for more details.
Setting `pad_token_id` to `eos_token_id`:151643 for open-end generation.
Both `max_new_tokens` (=2048) and `max_length`(=150) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


...

Setting `pad_token_id` to `eos_token_id`:151643 for open-end generation.
Both `max_new_tokens` (=2048) and `max_length`(=150) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


How to treat hypertension? The most common treatment for hypertension is to take medication. The most common medication for hypertension is a diuretic. The most common diuretic is furosemide. The most common side effect of furosemide is dizziness. The most common side effect of furosemide is dizziness. The most common side effect of furosemide is dizziness. The most common side effect of furosemide is dizziness. The most common side effect of furosemide is dizziness. The most common side effect of furosemide is dizziness. The most common side effect of furosemide is dizziness. The most common side effect of furosemide is dizziness. The most common side effect of furosemide is dizziness. The most common side effect of furosemide is dizziness. The most common side effect of furosemide is dizziness. The most common side effect of furosemide is dizziness. The most common side effect of furosemide is dizziness. The most common side effect of furosemide is dizziness. The most common side eff

Setting `pad_token_id` to `eos_token_id`:151643 for open-end generation.
Both `max_new_tokens` (=2048) and `max_length`(=150) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


Describe the pathophysiology of heart failure. Heart failure is a condition in which the heart is unable to pump enough blood to meet the body's needs. The pathophysiology of heart failure involves a complex interplay of factors that lead to the weakening of the heart muscle and the inability to pump effectively. Here are some key aspects of the pathophysiology of heart failure:

1. **Myocardial Hypertrophy**: The heart muscle becomes enlarged and thickened due to the accumulation of fatty deposits and scar tissue. This thickening can lead to reduced blood flow to the heart and impair its ability to pump blood effectively.

2. **Myocardial Fibrosis**: The heart muscle becomes scarred and thickened, which can further reduce blood flow and impair heart function.

3. **Cardiomyocyte Dysfunction**: The heart muscle cells (myocardium) become damaged and lose their ability to contract normally. This can lead to a decrease in the heart's ability to pump blood effectively.

4. **Cardiomyocyte 

: 